# Mega Project 1 — Intelligent Underwriting & Automated Credit Decisioning
## Problem 4: Credit Score Estimation — PDO Log-Odds Scaling of Problem 1's Real PD
## Model, with SHAP + LIME Explainability on the Loaded Champion

**Home Credit Default Risk — 6 Mega Projects Enterprise Suite**

### Business context
Convert the real probability-of-default (PD) that Notebook 01 (Problem 1) trained
into a single, human-readable internal credit score — the same log-odds scaling
convention (Points-to-Double-Odds / PDO) used across the retail credit industry
(FICO-style scorecards), so underwriters, risk committees, and downstream
decisioning systems can work with a 300–900 score instead of a raw probability.

### Dependency on Notebook 01 (Problem 1) — already the tightest coupling in this suite
This notebook **requires Notebook 01 to have already been run once** on your
machine — it loads Notebook 01's real trained champion model artifact
(`decision_engine/artifacts/notebook_01_champion_model.joblib`) rather than
training anything new. If that file is missing, this notebook raises a clear
error telling you to run Notebook 01 first (see Section 1 below), instead of
silently fabricating a model. Because this notebook's entire purpose is fidelity
to Problem 1's trained model, no change was needed to deepen that dependency —
it was already maximal. What changed this revision is a hardware-utilization
fix (below) and real SHAP + LIME explainability for the loaded champion model.

### Hardware-utilization fix (this revision — real root cause, not a hardware limit)
Same root cause and fix as every other notebook in this suite: the thread-count
environment variables were never actually being *set* anywhere, and heavy
libraries were imported before the ceiling was even computed. Fixed by calling
the shared `src/utils/performance_setup.py` module (HYPER) as the first
executable step, before any BLAS/OpenMP-reading library is imported, pinning
CPU affinity to every detected core, and adding Parquet-over-CSV caching
(shared with Notebook 01/02's cache under `decision_engine/_parquet_cache/`).

### SHAP + LIME explainability (new this revision)
Real SHAP (`TreeExplainer`, on a real 300-row sample of this notebook's own
scoring population) and LIME (3 representative real scored customers: most-
confident correct default flag, most-confident correct non-default, a real
misclassified customer) explainability for Notebook 01's loaded champion model —
supports every one of the top-4 candidate model types Notebook 01 could select
as champion. LIME's reference distribution is this notebook's own real scoring
population (documented explicitly as a stated substitute, since this notebook
has no direct access to Notebook 01's original training split).

### Feature parity guarantee (HYPER standard)
Both Notebook 01 (training) and this notebook (scoring) import the exact same
`engineer_credit_default_features()` function from the shared
`src/features/credit_default_features.py` module. This notebook additionally
runs a strict runtime equality check between the feature set it just built and
the exact feature list Notebook 01's model was trained on — if the shared
module is ever edited after Notebook 01 was trained, this notebook fails loudly
rather than silently scoring with a mismatched feature set.

### Data used (real, verified against your files before this notebook was written)
- `application_train.csv` — 307,511 real applications
- `bureau.csv` — 1,716,428 real external credit-bureau records
- Notebook 01's real trained model (`.joblib`) — the actual champion selected by
  mean cross-validated ROC-AUC on your machine's real run of Notebook 01

### What this notebook does (SOP Stages 1B–6 in one run)
1. Loads Notebook 01's real trained model + encoders/imputer.
2. Re-engineers the identical feature set on the real data via the shared module,
   and validates it matches Notebook 01's trained feature list exactly.
3. Runs real Exploratory Data Analysis & data-quality checks on the scoring
   population *before* scoring — missingness, IQR outliers, thin-file coverage,
   correlation with TARGET — with 3 vivid multicolor chart figures (SOP Stage 1B/2).
4. Computes real PD for every customer via `model.predict_proba`.
5. Applies PDO log-odds scorecard scaling — **three explicit, editable
   ASSUMPTION constants** (`BASE_SCORE=600`, `BASE_ODDS=50`, `PDO=20`), labeled
   with their source (standard scorecard convention, not derived from this data)
   — to convert PD into a 300–900 score.
6. Bins customers into 5 score bands (Very Poor → Excellent) and reports the
   **real** default rate per band from your actual `TARGET` column (not assumed
   to be monotonic — checked and reported, not forced).
7. Runs real Statistical Validation (SOP Stage 4): bootstrap 95% CI on the
   score's AUC, calibration-by-PD-decile, split-half PSI on the score
   distribution, and an explicit deployment readiness verdict.
8. Reports the thin-file population (customers with zero bureau history)
   separately, never silently blended into the main validation.
9. Computes real SHAP + LIME explainability for the loaded champion model on
   this notebook's own real scoring population.
10. Displays the real score distribution and real default-rate-by-band charts
   inline (vivid multicolor, per the standing chart-style rule), runs 11
   integrity self-checks (including new checks that SHAP values are finite,
   the SHAP feature count matches, LIME explanations were really computed, and
   the CPU thread ceiling was actually applied), then generates a full Stage-5
   reporting package — CSV outputs (including SHAP importances and LIME
   explanations as their own files), a colorized Word report with real
   narrative "stories" per chart, a 5-sheet Excel workbook with real
   conditional formatting and SMART-format insights, and an HTML dashboard
   with live slicers/filters over real precomputed alternate views — via the
   shared `src/reporting/report_builder.py` module (HYPER).
11. Saves a full run summary (including the real performance-config and
    explainability metadata) to `../decision_engine/artifacts/` (idempotent —
    overwritten in place every run, bit-identical on repeat runs against the
    same inputs).

### Standing rules this notebook follows
- **Zero-fabrication**: every score is computed live from Notebook 01's real
  model applied to your real data — the only non-data-derived inputs are the
  three labeled scorecard-scaling ASSUMPTION constants and two labeled
  illustrative-impact ASSUMPTION constants (Section 11) above.
- **WARP**: resource ceilings capped at 90% RAM / 95% CPU threads (never 100%,
  a safety ceiling — not a floor forced by padding), applied *before* any
  heavy import; CPU affinity pinned; Parquet-over-CSV caching; vivid
  multicolor charts throughout.
- **HYPER**: shared `src/features/`, `src/reporting/`, and `src/utils/`
  modules, built once, imported here and reused from Notebooks 01-02.
- **Privacy**: this notebook never prints your machine's absolute file paths.
- **Reproducibility**: `RANDOM_SEED = 42` fixed everywhere a random process is used.

### Before you run this
1. Run Notebook 01 first (produces the model artifact this notebook depends on).
2. Uses the same `project_config.json` as Notebook 01 (project root, `raw_data_dir`
   pointing at your real Kaggle CSV folder). On your real machine, `shap` and
   `lime` must be installed (`pip install shap lime`).

### Verification status
Verified end-to-end on a synthetic fixture matching the real schema via real
Jupyter execution (`jupyter nbconvert --execute`) — 0 errors, all 11 integrity
checks passed, HTML dashboard charts/filters confirmed rendering with 0 console
errors under a network-blocked Playwright check, Excel formulas confirmed
correct via LibreOffice headless recalculation. **Not yet run against your real
data.**


In [ ]:
# ============================================================================
# NOTEBOOK 03 — MEGA PROJECT 3: INTELLIGENT UNDERWRITING & AUTOMATED CREDIT
# DECISIONING | PROBLEM 4: CREDIT SCORE ESTIMATION
# Business Understanding, EDA, PDO Scorecard Scaling of Problem 1's Real PD
# Model, SHAP + LIME Explainability, Statistical Validation & Financial-Impact
# Reporting (SOP Stages 1-6 in one run)
# ----------------------------------------------------------------------------
# Zero-fabrication notice: every score below is computed live from Problem 1's
# REAL trained model applied to YOUR real data. The only non-data-derived inputs
# are the three scorecard-scaling constants (Section 6) and two illustrative
# cost/impact constants (Section 12), which are explicit, industry-standard or
# clearly labeled ASSUMPTIONs with a source note — never invented as real data.
# Re-running this cell always overwrites the same output paths (idempotent).
#
# INTERDEPENDENCY NOTE: this notebook is, by design, the single most tightly
# coupled notebook in Mega Project 1 — its entire purpose is to take Problem
# 1's real trained default-risk model and rescale its output into a real
# credit score. No change was needed here to deepen that dependency; it was
# already maximal. What changed this revision is a hardware-utilization fix
# (see below) and adding real SHAP + LIME explainability for the loaded
# champion model, applied to this notebook's own real scoring population.
#
# HARDWARE-UTILIZATION FIX (this revision): every BLAS/OpenMP thread-count
# environment variable is now set — via the shared, HYPER src/utils/
# performance_setup.py module, not a local duplicate — BEFORE numpy, polars,
# pandas, or scikit-learn are imported anywhere below. The PREVIOUS version of
# this file computed a thread ceiling but never actually set any environment
# variable, AND it imported all of those libraries at the very top of the
# file, before that ceiling was even computed — so no library ever saw a real
# ceiling regardless. See PERFORMANCE_SETUP_README.md / WARP notes.
# ============================================================================

import os
import sys
import json
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# SECTION 1 — Config + suite-root resolution (stdlib only — no heavy library
# is imported yet, deliberately, so the WARP thread ceiling below can be set
# before any of them read their thread-count environment variables. Never
# print the resolved raw-data path itself: this notebook may be shared
# publicly, e.g. on GitHub/Kaggle).
# ---------------------------------------------------------------------------
NOTEBOOK_DIR = Path.cwd()
def _find_suite_root(start: Path = None) -> Path:
    """Locate the home-credit-enterprise-suite project root (the folder containing
    project_config.json), regardless of where this notebook's kernel actually launched
    from. Checked in order, fastest and most explicit first -- deliberately NOT an
    unbounded/recursive filesystem scan (the exact "hangs / looks frozen" risk this
    suite's WARP performance module exists to avoid):
    1. HC_SUITE_ROOT environment variable, if set (see PERFORMANCE_SETUP_README.md)
    2. Walking UPWARD from the working directory (covers: cwd is this notebook's own
       mega_project_.../notebooks/ folder, the normal case when opened in place)
    3. A short list of well-known locations under the home directory (covers: the
       working directory being your home folder itself -- an ANCESTOR of the project,
       not inside it -- which an upward-only search cannot reach)
    """
    start = start or Path.cwd()
    marker = "project_config.json"
    env_override = os.environ.get("HC_SUITE_ROOT")
    if env_override and (Path(env_override) / marker).exists():
        return Path(env_override)
    for candidate in [start, *start.parents]:
        if (candidate / marker).exists():
            return candidate
    for candidate in [
        Path.home() / "Downloads" / "home-credit-enterprise-suite",
        Path.home() / "home-credit-enterprise-suite",
        Path.home() / "Desktop" / "home-credit-enterprise-suite",
        start / "home-credit-enterprise-suite",
        start / "Downloads" / "home-credit-enterprise-suite",
    ]:
        if (candidate / marker).exists():
            return candidate
    return None


SUITE_ROOT = _find_suite_root()
if SUITE_ROOT is None:
    raise FileNotFoundError(
        "project_config.json not found. Checked upward from the working directory plus "
        "well-known locations under your home folder. Fix: either open this notebook's "
        "own .ipynb file in place (rather than running its code in a fresh kernel "
        "elsewhere), or set an environment variable before launching Jupyter, e.g. on "
        'Windows PowerShell: $env:HC_SUITE_ROOT="C:\\Users\\rnand\\Downloads\\'
        'home-credit-enterprise-suite" -- see PERFORMANCE_SETUP_README.md.'
    )
config_path = SUITE_ROOT / "project_config.json"
with open(config_path) as f:
    CONFIG = json.load(f)

RAW_DIR = Path(CONFIG["raw_data_dir"])
SEED = int(CONFIG.get("random_seed", 42))
RANDOM_SEED = SEED

ARTIFACTS_DIR = SUITE_ROOT / "mega_project_1_underwriting_approval" / "decision_engine" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR = SUITE_ROOT / "mega_project_1_underwriting_approval" / "decision_engine" / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
PARQUET_CACHE_DIR = SUITE_ROOT / "mega_project_1_underwriting_approval" / "decision_engine" / "_parquet_cache"

sys.path.insert(0, str(SUITE_ROOT / "src"))
from utils.performance_setup import (
    configure_performance, pin_cpu_affinity, sklearn_n_jobs, gbm_thread_kwargs,
    threadpool_guard, free_memory, check_ram_headroom, load_csv_cached,
)
from utils.stats_checks import monotonic_within_noise

# ---------------------------------------------------------------------------
# SECTION 2 — WARP resource ceilings (hard cap, never 100%) — set BEFORE any
# of numpy/polars/pandas/scikit-learn are imported. configure_performance()
# sets OMP_NUM_THREADS / OPENBLAS_NUM_THREADS / MKL_NUM_THREADS /
# NUMEXPR_NUM_THREADS / POLARS_MAX_THREADS (etc.) as real OS environment
# variables — every one of those libraries reads its own copy of these
# exactly once, at its own import/init time, so this call MUST happen first.
# pin_cpu_affinity() additionally pins this process to every detected logical
# core, removing any pre-existing OS-level core restriction the env vars
# alone cannot fix.
# ---------------------------------------------------------------------------
PERF = configure_performance(
    ram_ceiling_fraction=float(CONFIG.get("ram_ceiling_fraction", 0.90)),
    cpu_ceiling_fraction=float(CONFIG.get("cpu_ceiling_fraction", 0.95)),
)
pin_cpu_affinity(PERF)
TOTAL_RAM_GB = PERF["total_ram_gb"]
TOTAL_THREADS = PERF["logical_cores"]
RAM_CEILING_GB = PERF["ram_ceiling_gb"]
CPU_CEILING_THREADS = PERF["n_threads"]

# ---------------------------------------------------------------------------
# SECTION 3 — Heavy-library imports (deliberately AFTER Section 2 above)
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import joblib
from sklearn.metrics import roc_auc_score
import shap
from lime.lime_tabular import LimeTabularExplainer

np.random.seed(SEED)
T0 = time.time()

from features.credit_default_features import engineer_credit_default_features
from reporting.report_builder import (
    write_csv_outputs, build_word_report, build_excel_workbook,
    build_html_dashboard, assumption_ref, VIVID_PALETTE, _palette,
)
# Standing chart-style rule (all problems): vivid multicoloured charts everywhere,
# using the 8-hue CVD-validated categorical palette canonicalized in
# src/reporting/report_builder.py (HYPER -- imported, not redefined per notebook).

UPSTREAM_MODEL_PATH = ARTIFACTS_DIR / "notebook_01_champion_model.joblib"
if not UPSTREAM_MODEL_PATH.exists():
    raise FileNotFoundError(
        "This notebook depends on Notebook 01 (Problem 1: Credit Default Prediction). "
        f"Expected trained model artifact not found at {UPSTREAM_MODEL_PATH.name} "
        "(relative to decision_engine/artifacts/). Run Notebook 01 first, then re-run this cell."
    )

print(f"[WARP] {TOTAL_RAM_GB:.1f} GB RAM / {TOTAL_THREADS} threads detected -> "
      f"ceiling {RAM_CEILING_GB} GB RAM, {CPU_CEILING_THREADS} threads "
      f"(env vars applied before any heavy import; CPU affinity pinned to all cores)")
print("[DATA] Raw data directory resolved and verified (path withheld from output by design).")
print(f"[SEED] RANDOM_SEED = {SEED}")

# ---------------------------------------------------------------------------
# SECTION 4 — Load Problem 1's real trained model + real data (WARP:
# Parquet-over-CSV cache, shared with Notebook 01/02 under the same
# decision_engine/_parquet_cache/ directory), re-engineer the IDENTICAL
# feature set via the shared module (guarantees column-for-column match with
# what the model was actually trained on).
#
# Scope note (deliberate, not a gap): Problem 4's entire purpose is fidelity to
# Problem 1's trained model, so it intentionally reuses Problem 1's exact real
# feature set (application_train + bureau, via the same shared
# src/features/credit_default_features.py module) rather than re-deriving a
# separate, larger feature set the way Problem 3 (Notebook 02) does for its own
# from-scratch model. This is verified at runtime below (feature-set match
# check), not assumed.
# ---------------------------------------------------------------------------
bundle = joblib.load(UPSTREAM_MODEL_PATH)
model = bundle["model"]
ord_enc = bundle["ordinal_encoder"]
imputer = bundle["imputer"]
FEATURE_COLS = bundle["feature_cols"]
NUMERIC_FEATURES = bundle["numeric_features"]
CATEGORICAL_FEATURES = bundle["categorical_features"]
UPSTREAM_CHAMPION = bundle["champion_name"]
print(f"[UPSTREAM] Loaded Notebook 01's real champion model: {UPSTREAM_CHAMPION}")

app = load_csv_cached(RAW_DIR / "application_train.csv", PARQUET_CACHE_DIR, null_values=["", "NA", "XNA"])
bureau = load_csv_cached(RAW_DIR / "bureau.csv", PARQUET_CACHE_DIR, null_values=["", "NA", "XNA"])
check_ram_headroom(PERF)
df, feat_numeric_check, feat_cat_check = engineer_credit_default_features(app, bureau)
if feat_numeric_check != NUMERIC_FEATURES or feat_cat_check != CATEGORICAL_FEATURES:
    raise ValueError(
        "Feature set built here does not match Notebook 01's trained feature set. "
        "This means the shared feature module changed after Notebook 01 was trained -- "
        "re-run Notebook 01 to retrain against the current feature set before continuing."
    )
N_SCORE_POP = df.height
print(f"[SCORE POPULATION] {N_SCORE_POP:,} real customers to be scored")

# ---------------------------------------------------------------------------
# SECTION 5 — Exploratory Data Analysis & Data Quality of the Scoring
# Population (SOP Stage 1B/2). Computed on `df` -- the real engineered feature
# set this notebook is about to score -- before any encoding/imputation below,
# so every chart shows genuine pre-imputation missingness, not a cleaned view
# dressed up as "before".
# ---------------------------------------------------------------------------
null_counts = df.select(FEATURE_COLS).null_count().to_pandas().T.reset_index()
null_counts.columns = ["column", "n_null"]
null_counts["pct_null"] = null_counts["n_null"] / N_SCORE_POP
null_counts = null_counts[null_counts["n_null"] > 0].sort_values("pct_null", ascending=False)
top_missing = null_counts.head(15)
print(f"[EDA] {len(null_counts)} / {len(FEATURE_COLS)} real scoring features have at least one "
      f"missing value (imputed via Notebook 01's fitted imputer below, never dropped). Top 5:")
for _, row in null_counts.head(5).iterrows():
    print(f"  {row['column']}: {row['pct_null']:.2%} ({int(row['n_null']):,} rows)")

if "BUREAU_CNT_CREDITS" in df.columns:
    n_thin = int((df["BUREAU_CNT_CREDITS"] <= 0).sum())
    thin_pct = n_thin / N_SCORE_POP if N_SCORE_POP else 0.0
else:
    n_thin, thin_pct = 0, 0.0
print(f"[EDA] Thin-file population (zero real bureau credit records): {n_thin:,} / {N_SCORE_POP:,} "
      f"({thin_pct:.2%}) -- scored via application-only signal, tracked separately throughout.")

DIST_COLS = [c for c in ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AGE_YEARS"] if c in df.columns]
dist_data = {c: df[c].drop_nulls().to_numpy() for c in DIST_COLS}
OUTLIER_SUMMARY = []
for c in DIST_COLS:
    vals = dist_data[c]
    q1, q3 = np.percentile(vals, [25, 75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = int(((vals < lo) | (vals > hi)).sum())
    OUTLIER_SUMMARY.append({"column": c, "n_outliers_iqr": n_out, "pct_outliers_iqr": n_out / len(vals),
                             "iqr_lower": float(lo), "iqr_upper": float(hi)})
    print(f"[EDA] IQR outliers in {c}: {n_out:,} ({n_out / len(vals):.2%}) outside [{lo:,.0f}, {hi:,.0f}]")

CORR_COLS = [c for c in NUMERIC_FEATURES if c in df.columns][:15]
corr_pdf = df.select(["TARGET"] + CORR_COLS).to_pandas()
target_corr = corr_pdf.corr(numeric_only=True)["TARGET"].drop("TARGET").sort_values()
print(f"[EDA] Real correlation of scoring features with TARGET (top 3 by |r|): "
      f"{target_corr.abs().sort_values(ascending=False).head(3).round(4).to_dict()}")

# --- EDA Figure 1: Data Quality Overview (2x2, vivid multicolor) -----------
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

axes[0, 0].barh(top_missing["column"][::-1], (top_missing["pct_null"][::-1] * 100),
                 color=_palette(len(top_missing)))
axes[0, 0].set_xlabel("% Missing"); axes[0, 0].set_title(f"Top {len(top_missing)} Scoring Features by Real Missing %")

axes[0, 1].bar(["Bureau-Scored", "Thin-File (App-Only)"], [N_SCORE_POP - n_thin, n_thin],
                color=[VIVID_PALETTE[0], VIVID_PALETTE[3]])
axes[0, 1].set_title(f"Real Bureau-Coverage Split (thin-file rate {thin_pct:.2%})")
for i, v in enumerate([N_SCORE_POP - n_thin, n_thin]):
    axes[0, 1].text(i, v, f"{v:,}", ha="center", va="bottom")

if "AMT_INCOME_TOTAL" in dist_data:
    axes[1, 0].hist(dist_data["AMT_INCOME_TOTAL"], bins=40, color=VIVID_PALETTE[1], edgecolor="white")
    axes[1, 0].set_title("Real AMT_INCOME_TOTAL Distribution (scoring population)")
if "AGE_YEARS" in dist_data:
    axes[1, 1].hist(dist_data["AGE_YEARS"], bins=40, color=VIVID_PALETTE[2], edgecolor="white")
    axes[1, 1].set_title("Real Applicant Age Distribution (scoring population)")

plt.tight_layout()
eda_overview_path = REPORTS_DIR / "notebook_03_eda_overview.png"
plt.savefig(eda_overview_path, dpi=110)
plt.show()

# --- EDA Figure 2: Numeric Distributions & IQR Outliers --------------------
fig, axes = plt.subplots(2, len(DIST_COLS), figsize=(5 * len(DIST_COLS), 8))
for i, c in enumerate(DIST_COLS):
    axes[0, i].hist(dist_data[c], bins=40, color=VIVID_PALETTE[i % len(VIVID_PALETTE)], edgecolor="white")
    axes[0, i].set_title(f"Real Distribution: {c}")
    bx = axes[1, i].boxplot(dist_data[c], vert=False, patch_artist=True)
    bx["boxes"][0].set_facecolor(VIVID_PALETTE[(i + 3) % len(VIVID_PALETTE)])
    n_out = next(o["n_outliers_iqr"] for o in OUTLIER_SUMMARY if o["column"] == c)
    axes[1, i].set_title(f"IQR Outliers: {n_out:,} ({n_out / len(dist_data[c]):.1%})")
plt.tight_layout()
eda_distributions_path = REPORTS_DIR / "notebook_03_eda_distributions.png"
plt.savefig(eda_distributions_path, dpi=110)
plt.show()

# --- EDA Figure 3: Real correlation of scoring features with TARGET --------
fig, ax = plt.subplots(figsize=(9, 6.5))
colors = [VIVID_PALETTE[0] if v >= 0 else VIVID_PALETTE[7] for v in target_corr]
ax.barh(target_corr.index, target_corr.values, color=colors)
ax.axvline(0, color="#898781", linewidth=1)
ax.set_title("Real Correlation of Scoring Features with TARGET\n(blue = positive, red = negative)")
plt.tight_layout()
eda_correlation_path = REPORTS_DIR / "notebook_03_eda_correlation.png"
plt.savefig(eda_correlation_path, dpi=110)
plt.show()

EDA_CHART_PATHS = [eda_overview_path, eda_distributions_path, eda_correlation_path]

# ---------------------------------------------------------------------------
# SECTION 6 — Score the real population with Problem 1's real model, then
# apply PDO log-odds scorecard scaling (explicit ASSUMPTIONs, industry-standard
# scorecard convention, NOT derived from this data).
# ---------------------------------------------------------------------------
pdf = df.select(["SK_ID_CURR", "TARGET"] + FEATURE_COLS).to_pandas()
for c in CATEGORICAL_FEATURES:
    pdf[c] = pdf[c].astype(object).fillna("Missing").astype(str).astype("category")
for c in NUMERIC_FEATURES:
    pdf[c] = pdf[c].astype("float32")

X = pdf[FEATURE_COLS].copy()
if CATEGORICAL_FEATURES:
    X[CATEGORICAL_FEATURES] = ord_enc.transform(pdf[CATEGORICAL_FEATURES].astype(str))
X[NUMERIC_FEATURES] = imputer.transform(pdf[NUMERIC_FEATURES])

PD = model.predict_proba(X)[:, 1]
PD = np.clip(PD, 1e-6, 1 - 1e-6)  # guard against exact 0/1 before log-odds
print(f"[PD] Real PD distribution from Problem 1's model: "
      f"min={PD.min():.4f} mean={PD.mean():.4f} max={PD.max():.4f}")

# ASSUMPTION (source: standard credit-scorecard convention, e.g. FICO-style scaling):
#   BASE_SCORE = 600, BASE_ODDS = 50 (good:bad), PDO = 20 (points to double the odds)
BASE_SCORE = 600.0
BASE_ODDS = 50.0
PDO = 20.0
FACTOR = PDO / np.log(2)
OFFSET = BASE_SCORE - FACTOR * np.log(BASE_ODDS)

odds = (1 - PD) / PD
raw_score = OFFSET + FACTOR * np.log(odds)
SCORE = np.clip(raw_score, 300, 900)  # FICO-style display range, also an ASSUMPTION
pdf["PD"] = PD
pdf["CREDIT_SCORE"] = SCORE

print(f"[SCALING] BASE_SCORE={BASE_SCORE}, BASE_ODDS={BASE_ODDS}, PDO={PDO} "
      f"(ASSUMPTION, source: standard scorecard convention) -> "
      f"Factor={FACTOR:.3f}, Offset={OFFSET:.3f}")
print(f"[SCORE] Real computed score distribution: "
      f"min={SCORE.min():.0f} p25={np.percentile(SCORE,25):.0f} median={np.median(SCORE):.0f} "
      f"p75={np.percentile(SCORE,75):.0f} max={SCORE.max():.0f}")

# ---------------------------------------------------------------------------
# SECTION 7 — Score bands + real validation against actual TARGET
# ---------------------------------------------------------------------------
BAND_LABELS = ["Very Poor", "Poor", "Fair", "Good", "Excellent"]
# .astype(str) immediately after qcut, deliberately: pd.qcut(..., labels=...) returns a
# pandas Categorical dtype column, and a real cross-pandas-version bug was observed where
# a later .map(...).fillna(0.0) on a Categorical-typed column (Section 11 below) raises
# "TypeError: Cannot setitem on a Categorical with a new category (0.0), set the categories
# first" on some pandas versions (map-on-Categorical preserves the Categorical dtype there)
# while working fine on others (map-on-Categorical returns a plain float64 there). Casting
# to plain str once, here, at the source, removes the Categorical dtype for every downstream
# use of this column (groupby, map, JSON export) so this notebook behaves identically across
# pandas versions, rather than patching each downstream symptom individually.
pdf["SCORE_BAND"] = pd.qcut(pdf["CREDIT_SCORE"], q=5, labels=BAND_LABELS, duplicates="drop").astype(str)

band_validation = (
    pdf.groupby("SCORE_BAND", observed=True)
    .agg(n_customers=("TARGET", "size"), real_default_rate=("TARGET", "mean"),
         mean_score=("CREDIT_SCORE", "mean"))
    .reset_index()
)
print("[VALIDATION] Real default rate by score band (should decrease as score rises):")
print(band_validation.to_string(index=False))

band_order = [b for b in BAND_LABELS if b in band_validation["SCORE_BAND"].astype(str).values]
_bv_indexed = band_validation.set_index("SCORE_BAND").loc[band_order]
rates_in_order = _bv_indexed["real_default_rate"].tolist()
counts_in_order = _bv_indexed["n_customers"].tolist()
# Statistically-principled monotonicity check -- same fix, same rationale, as
# Notebook 04's identical pattern (see src/utils/stats_checks.py and
# CHANGELOG.md [1.0.2]): a real, Bonferroni-corrected two-proportion z-test
# per adjacent band pair, not a strict zero-tolerance ordering check.
is_monotonic_decreasing, _monotonicity_detail = monotonic_within_noise(rates_in_order, counts_in_order, alpha=0.05)
print(f"[VALIDATION] Score-to-default monotonicity holds (within statistical noise): {is_monotonic_decreasing}")
for _row in _monotonicity_detail:
    if _row["reversed"]:
        print(f"  [MONOTONICITY] pair {_row['pair_index']} ({band_order[_row['pair_index']]} -> "
              f"{band_order[_row['pair_index'] + 1]}) reversed: z={_row['z']:.4f}, p={_row['p_value']:.4f}, "
              f"significant={_row['statistically_significant_reversal']} "
              f"(alpha={_row['alpha_used_bonferroni_corrected']:.4f}, Bonferroni-corrected)")

# ---------------------------------------------------------------------------
# SECTION 7B — SHAP Explainability (Notebook 01's loaded champion model,
# applied to this notebook's own real scoring population). shap.TreeExplainer
# supports every one of the top-4 candidate model types Notebook 01 could
# have selected as champion (RandomForest, XGBoost, CatBoost, LightGBM).
# ---------------------------------------------------------------------------
SHAP_SAMPLE_N = min(300, len(X))
_shap_sample_idx = X.sample(n=SHAP_SAMPLE_N, random_state=SEED).index
X_shap_sample = X.loc[_shap_sample_idx]
shap_explainer = shap.TreeExplainer(model)
_raw_shap = shap_explainer.shap_values(X_shap_sample)
if isinstance(_raw_shap, list):
    SHAP_VALUES = np.asarray(_raw_shap[1])          # positive ("Default") class
elif isinstance(_raw_shap, np.ndarray) and _raw_shap.ndim == 3:
    SHAP_VALUES = _raw_shap[:, :, 1]                 # (rows, features, classes) -> positive class
else:
    SHAP_VALUES = np.asarray(_raw_shap)

mean_abs_shap = np.abs(SHAP_VALUES).mean(axis=0)
shap_importance_df = pd.DataFrame(
    {"feature": FEATURE_COLS, "mean_abs_shap": mean_abs_shap}
).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
TOP_SHAP_FEATURES = shap_importance_df.head(15)

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(TOP_SHAP_FEATURES["feature"][::-1], TOP_SHAP_FEATURES["mean_abs_shap"][::-1],
        color=_palette(len(TOP_SHAP_FEATURES)))
ax.set_xlabel("Mean |SHAP value| (real impact on predicted default probability)")
ax.set_title(f"SHAP Feature Importance — Upstream Champion ({UPSTREAM_CHAMPION}), "
             f"real sample of {SHAP_SAMPLE_N} scored customers")
plt.tight_layout()
shap_summary_path = REPORTS_DIR / "notebook_03_shap_summary.png"
plt.savefig(shap_summary_path, dpi=110)
plt.show()

shap_beeswarm_path = None
try:
    plt.figure(figsize=(9, 7))
    shap.summary_plot(SHAP_VALUES, X_shap_sample, feature_names=FEATURE_COLS, show=False, max_display=15)
    plt.tight_layout()
    shap_beeswarm_path = REPORTS_DIR / "notebook_03_shap_beeswarm.png"
    plt.savefig(shap_beeswarm_path, dpi=110)
    plt.close()
except Exception as e:
    print(f"[SHAP] Beeswarm detail plot skipped ({type(e).__name__}: {e}); "
          f"the bar-chart summary above is unaffected — explainability chart only, no correctness impact.")
print(f"[SHAP] Real SHAP explainability computed for upstream champion {UPSTREAM_CHAMPION} on a real "
      f"{SHAP_SAMPLE_N}-row sample of the scoring population. Top real driver: "
      f"{TOP_SHAP_FEATURES.iloc[0]['feature']} (mean |SHAP|={TOP_SHAP_FEATURES.iloc[0]['mean_abs_shap']:.4f}).")

# ---------------------------------------------------------------------------
# SECTION 7C — LIME Explainability (Notebook 01's loaded champion model).
# Local, instance-level explanations on real representative scored customers
# selected from this run's own real predictions -- never fabricated.
#
# Design note: LIME's `training_data` parameter is used only to compute
# per-feature perturbation statistics, not to retrain anything. This notebook
# does not have direct access to Notebook 01's original training split, so it
# uses its own real, freshly-encoded scoring population (`X`) as that
# reference distribution -- a reasonable, explicitly-stated substitute given
# this notebook's scope (scoring, not training).
# ---------------------------------------------------------------------------
CATEGORICAL_FEATURE_IDX = [FEATURE_COLS.index(c) for c in CATEGORICAL_FEATURES]
lime_explainer = LimeTabularExplainer(
    training_data=X.to_numpy(),
    feature_names=FEATURE_COLS,
    categorical_features=CATEGORICAL_FEATURE_IDX,
    class_names=["No Default", "Default"],
    mode="classification",
    random_state=SEED,
)

_score_pred = (PD >= 0.5).astype(int)
_lime_pool = pd.DataFrame({
    "idx": np.arange(len(X)), "actual": pdf["TARGET"].to_numpy(),
    "proba": PD, "pred": _score_pred,
})
_lime_cases = {}
_correct_default = _lime_pool[(_lime_pool.actual == 1) & (_lime_pool.pred == 1)]
if len(_correct_default):
    _lime_cases["Most-confident correct default flag"] = int(_correct_default.loc[_correct_default.proba.idxmax(), "idx"])
_correct_nondefault = _lime_pool[(_lime_pool.actual == 0) & (_lime_pool.pred == 0)]
if len(_correct_nondefault):
    _lime_cases["Most-confident correct non-default"] = int(_correct_nondefault.loc[_correct_nondefault.proba.idxmin(), "idx"])
_misclassified = _lime_pool[_lime_pool.actual != _lime_pool.pred]
if len(_misclassified):
    _lime_cases["A real misclassified customer"] = int(_misclassified.sample(n=1, random_state=SEED)["idx"].iloc[0])

LIME_EXPLANATIONS = []
_lime_fig_paths = []
for case_label, row_idx in _lime_cases.items():
    instance = X.iloc[row_idx].to_numpy()
    exp = lime_explainer.explain_instance(instance, model.predict_proba, num_features=8)
    for feat, weight in exp.as_list():
        LIME_EXPLANATIONS.append({"case": case_label, "feature_condition": feat, "weight": float(weight)})
    try:
        fig_l = exp.as_pyplot_figure()
        fig_l.suptitle(f"LIME — {case_label} (real scored customer)")
        fig_l.tight_layout()
        p = REPORTS_DIR / f"notebook_03_lime_{len(_lime_fig_paths)}.png"
        fig_l.savefig(p, dpi=110)
        plt.close(fig_l)
        _lime_fig_paths.append(p)
    except Exception as e:
        print(f"[LIME] Plot skipped for '{case_label}' ({type(e).__name__}: {e}); table rows above are unaffected.")

lime_explanations_df = pd.DataFrame(LIME_EXPLANATIONS)
lime_panel_path = _lime_fig_paths[0] if _lime_fig_paths else None
print(f"[LIME] Real local explanations computed for upstream champion {UPSTREAM_CHAMPION} on "
      f"{len(_lime_cases)} representative real scored case(s): {list(_lime_cases.keys())}")

# ---------------------------------------------------------------------------
# SECTION 8 — Statistical Validation & Deployment Readiness (SOP Stage 4)
# Bootstrap AUC confidence interval on the real PD's discrimination, calibration
# by PD decile, split-half PSI on the real SCORE distribution, plus the real
# monotonicity result from Section 7 -- every figure computed live.
# ---------------------------------------------------------------------------
y_arr = pdf["TARGET"].to_numpy()
rng = np.random.default_rng(SEED)
N_BOOTSTRAP = 1000
boot_aucs = []
for _ in range(N_BOOTSTRAP):
    idx = rng.integers(0, len(y_arr), len(y_arr))
    y_bs = y_arr[idx]
    if len(np.unique(y_bs)) < 2:
        continue
    boot_aucs.append(roc_auc_score(y_bs, PD[idx]))
boot_aucs = np.array(boot_aucs)
AUC_CI_LOW, AUC_CI_HIGH = (float(np.percentile(boot_aucs, 2.5)), float(np.percentile(boot_aucs, 97.5))) \
    if len(boot_aucs) > 0 else (float("nan"), float("nan"))
REAL_SCORE_AUC = float(roc_auc_score(y_arr, PD))
print(f"[VALIDATION] Real score discrimination (AUC of PD vs actual TARGET): {REAL_SCORE_AUC:.4f} "
      f"({len(boot_aucs)}-resample bootstrap 95% CI [{AUC_CI_LOW:.4f}, {AUC_CI_HIGH:.4f}])")

calib_df = pd.DataFrame({"y": y_arr, "p": PD})
calib_df["decile"] = pd.qcut(calib_df["p"], q=10, labels=False, duplicates="drop")
calib_summary = (
    calib_df.groupby("decile")
    .agg(mean_predicted=("p", "mean"), actual_rate=("y", "mean"), n=("y", "size"))
    .reset_index()
)
MEAN_CALIBRATION_GAP = float((calib_summary["mean_predicted"] - calib_summary["actual_rate"]).abs().mean())
print(f"[VALIDATION] Real mean calibration gap (|predicted - actual| across "
      f"{len(calib_summary)} PD deciles): {MEAN_CALIBRATION_GAP:.4f}")

half_idx = rng.permutation(len(SCORE))
half_a = SCORE[half_idx[: len(half_idx) // 2]]
half_b = SCORE[half_idx[len(half_idx) // 2:]]
_bins = np.linspace(300, 900, 11)
def _psi(a, b, bins):
    a_counts, _ = np.histogram(a, bins=bins)
    b_counts, _ = np.histogram(b, bins=bins)
    a_pct = np.clip(a_counts / max(a_counts.sum(), 1), 1e-4, None)
    b_pct = np.clip(b_counts / max(b_counts.sum(), 1), 1e-4, None)
    return float(np.sum((a_pct - b_pct) * np.log(a_pct / b_pct)))
SPLIT_HALF_PSI = _psi(half_a, half_b, _bins)
print(f"[VALIDATION] Real split-half PSI on the SCORE distribution: {SPLIT_HALF_PSI:.4f}")

CALIBRATION_GAP_THRESHOLD = 0.10   # ASSUMPTION — informal convention: <0.10 mean gap considered acceptable
PSI_STABILITY_THRESHOLD = 0.10     # ASSUMPTION — standard PSI convention: <0.10 stable, 0.10-0.25 moderate shift, >0.25 significant shift
deployment_checks = [
    ("auc_ci_lower_above_random", AUC_CI_LOW > 0.5),
    ("mean_calibration_gap_acceptable", MEAN_CALIBRATION_GAP < CALIBRATION_GAP_THRESHOLD),
    ("split_half_psi_stable", SPLIT_HALF_PSI < PSI_STABILITY_THRESHOLD),
    ("score_monotonicity_holds", is_monotonic_decreasing),
]
DEPLOYMENT_READY = all(ok for _, ok in deployment_checks)
_failed_deployment_checks = [name for name, ok in deployment_checks if not ok]
# NOTE: see the identical comment in pipeline_body.py (Notebook 01) -- this
# "deployment_checks" family is a separate, stricter STATISTICAL ROBUSTNESS
# gate from the "integrity_checks" structural pipeline-sanity family reported
# later in this notebook. Fixed during the hardening pass (see CHANGELOG.md).
DEPLOYMENT_VERDICT = (
    "RECOMMENDED FOR PRODUCTION" if DEPLOYMENT_READY
    else "NOT RECOMMENDED FOR PRODUCTION YET — failed: " + ", ".join(_failed_deployment_checks) +
         " (this is a separate, stricter statistical-robustness gate, distinct from the "
         "structural pipeline integrity checks reported elsewhere in this notebook's output; "
         "failing here does not indicate a code defect, and passing all integrity checks does "
         "not imply this gate passed -- expected and informational on small or noisy "
         "real/synthetic samples, see this problem's MODEL_CARD.md)"
)
for name, ok in deployment_checks:
    print(f"[VALIDATION-CHECK] {name}: {'PASS' if ok else 'FAIL'}")
print(f"[VALIDATION] Deployment readiness verdict: {DEPLOYMENT_VERDICT}")

# ---------------------------------------------------------------------------
# SECTION 9 — Inline charts (vivid multicolor, per the standing chart-style rule)
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].hist(pdf["CREDIT_SCORE"], bins=40, color=VIVID_PALETTE[0])
axes[0].set_xlabel("Credit Score"); axes[0].set_ylabel("Customers"); axes[0].set_title("Real Score Distribution")
band_colors = _palette(len(band_validation))
axes[1].bar(band_validation["SCORE_BAND"].astype(str), band_validation["real_default_rate"], color=band_colors)
axes[1].set_ylabel("Real Default Rate"); axes[1].set_title("Real Default Rate by Score Band")
plt.setp(axes[1].get_xticklabels(), rotation=30, ha="right")
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "notebook_03_score_distribution.png", dpi=110)
plt.show()

# ---------------------------------------------------------------------------
# SECTION 10 — Integrity self-checks (fail loudly, never silently pass bad state)
# ---------------------------------------------------------------------------
checks = [
    ("upstream_model_loaded", model is not None),
    ("feature_set_matches_upstream", feat_numeric_check == NUMERIC_FEATURES and feat_cat_check == CATEGORICAL_FEATURES),
    ("all_customers_scored", pdf["CREDIT_SCORE"].notna().all()),
    ("score_within_declared_range", pdf["CREDIT_SCORE"].between(300, 900).all()),
    ("no_pd_out_of_bounds", bool(np.all((PD > 0) & (PD < 1)))),
    ("band_count_correct", pdf["SCORE_BAND"].nunique() <= 5),
    ("score_auc_above_random", REAL_SCORE_AUC > 0.5),
    ("shap_values_finite", bool(np.isfinite(SHAP_VALUES).all())),
    ("shap_feature_count_matches", len(shap_importance_df) == len(FEATURE_COLS)),
    ("lime_explanations_computed", len(LIME_EXPLANATIONS) > 0),
    ("cpu_thread_ceiling_applied_before_import",
     os.environ.get("OMP_NUM_THREADS") == str(CPU_CEILING_THREADS)),
]
for name, ok in checks:
    print(f"[CHECK] {name}: {'PASS' if ok else 'FAIL'}")
failed = [n for n, ok in checks if not ok]
if failed:
    raise AssertionError(f"Integrity checks failed: {failed}")

# ---------------------------------------------------------------------------
# SECTION 11 — Financial-Impact Reporting & Packaging (SOP Stage 5)
# Real CSV outputs + a Word report + an Excel workbook (formula-driven) + an
# HTML dashboard, generated from this run's own real computed results via the
# shared src/reporting module (HYPER: built once, reused by every notebook).
# ---------------------------------------------------------------------------
if "AMT_CREDIT" in pdf.columns:
    portfolio_by_band = pdf.groupby("SCORE_BAND", observed=True)["AMT_CREDIT"].sum().reindex(band_order)
    TOTAL_PORTFOLIO_VOLUME = float(pdf["AMT_CREDIT"].sum())
else:
    portfolio_by_band = pd.Series(0.0, index=band_order)
    TOTAL_PORTFOLIO_VOLUME = 0.0

ASSUMPTIONS = {
    "AVG_MANUAL_SCORING_COST_PER_CUSTOMER": 3.0,
}
ASSUMPTION_NOTES = {
    "AVG_MANUAL_SCORING_COST_PER_CUSTOMER": "Illustrative operations-cost convention for a manual credit-scoring "
                                             "review, applied only to the automated-scoring count, never blended "
                                             "into the real portfolio volume figures above",
}
ESTIMATED_MANUAL_SCORING_COST_AVOIDED = float(N_SCORE_POP) * ASSUMPTIONS["AVG_MANUAL_SCORING_COST_PER_CUSTOMER"]
print(f"[IMPACT] Real total scored portfolio volume (AMT_CREDIT): ${TOTAL_PORTFOLIO_VOLUME:,.0f} across "
      f"{N_SCORE_POP:,} real customers.")

score_export_df = pdf[["SK_ID_CURR", "PD", "CREDIT_SCORE", "SCORE_BAND", "TARGET"]].copy()
band_validation_export = band_validation.copy()
band_validation_export["portfolio_amt_credit"] = band_validation_export["SCORE_BAND"].map(portfolio_by_band).fillna(0.0)
calibration_df = calib_summary.rename(columns={"decile": "score_decile"})

_worst_band = band_validation.sort_values("real_default_rate", ascending=False).iloc[0]
_best_band = band_validation.sort_values("real_default_rate", ascending=True).iloc[0]
STORY_SCORE_CHART = [
    f"The real computed score distribution spans {SCORE.min():.0f}-{SCORE.max():.0f} "
    f"(median {np.median(SCORE):.0f}), scaled from Problem 1's real {UPSTREAM_CHAMPION} PD model via a "
    f"standard PDO log-odds scorecard (base {BASE_SCORE:.0f}, {PDO:.0f} points to double the odds).",
    f"Real discrimination: AUC {REAL_SCORE_AUC:.4f} (95% bootstrap CI [{AUC_CI_LOW:.4f}, {AUC_CI_HIGH:.4f}]) "
    f"against actual TARGET outcomes.",
    f"Deployment readiness verdict: {DEPLOYMENT_VERDICT}.",
]
STORY_BAND_CHART = [
    f"'{_worst_band['SCORE_BAND']}' carries the highest real default rate ({_worst_band['real_default_rate']:.2%}, "
    f"mean score {_worst_band['mean_score']:.0f}); '{_best_band['SCORE_BAND']}' the lowest "
    f"({_best_band['real_default_rate']:.2%}, mean score {_best_band['mean_score']:.0f}).",
    f"Score-to-default monotonicity {'holds' if is_monotonic_decreasing else 'does not fully hold'} across "
    f"all {len(band_validation)} bands (informational — reported, not forced).",
    "Switch the view above to see the same 5 bands by real customer count instead of real default rate.",
]
STORY_SHAP_CHART = [
    f"Real SHAP explainability was computed for Notebook 01's loaded champion model ({UPSTREAM_CHAMPION}) on a "
    f"real {SHAP_SAMPLE_N}-row sample of this notebook's own scoring population.",
    f"The single strongest real driver of predicted default probability is "
    f"{TOP_SHAP_FEATURES.iloc[0]['feature']} (mean |SHAP| = {TOP_SHAP_FEATURES.iloc[0]['mean_abs_shap']:.4f}), "
    f"followed by {TOP_SHAP_FEATURES.iloc[1]['feature']} "
    f"(mean |SHAP| = {TOP_SHAP_FEATURES.iloc[1]['mean_abs_shap']:.4f}).",
    "Positive SHAP values push a real customer's predicted default probability (and therefore lower credit "
    "score) up; negative values push it down — see the beeswarm detail chart for the real direction of each "
    "feature's effect.",
]
STORY_LIME_CHART = [
    f"Real local (instance-level) LIME explanations were computed for {len(_lime_cases)} representative real "
    f"scored customers selected from this run's own predictions: {', '.join(_lime_cases.keys())}.",
    "Unlike SHAP's global feature-importance ranking above, LIME shows exactly which real feature values "
    "moved THIS SPECIFIC customer's score — useful for explaining an individual credit-score result to a "
    "customer or auditor.",
    "Full per-feature weights for every case are in the LIME Instance Explanations table/sheet below.",
]
STORY_MISSING_CHART = [
    f"{len(null_counts)} of {len(FEATURE_COLS)} real scoring features have at least one missing value; the "
    f"worst, {top_missing.iloc[0]['column'] if len(top_missing) else 'n/a'}, is missing in "
    f"{(top_missing.iloc[0]['pct_null'] if len(top_missing) else 0):.1%} of the scoring population.",
    f"{n_thin:,} / {N_SCORE_POP:,} customers ({thin_pct:.2%}) are thin-file (zero real bureau credit records) "
    f"and are scored on application-only signal — tracked separately throughout, never silently merged.",
    "Use the filter above to switch between the top-15 and top-5 views of this same real ranking.",
]
STORY_CALIB_CHART = [
    f"Mean calibration gap across {len(calib_summary)} real PD deciles is {MEAN_CALIBRATION_GAP:.4f} "
    f"(ASSUMPTION threshold: <{CALIBRATION_GAP_THRESHOLD}), so the score is "
    f"{'well-calibrated' if MEAN_CALIBRATION_GAP < CALIBRATION_GAP_THRESHOLD else 'not yet well-calibrated'}.",
    f"Split-half PSI on the real SCORE distribution is {SPLIT_HALF_PSI:.4f} "
    f"(ASSUMPTION threshold: <{PSI_STABILITY_THRESHOLD}), indicating "
    f"{'a stable' if SPLIT_HALF_PSI < PSI_STABILITY_THRESHOLD else 'a shifting'} score distribution.",
]

DEPLOY_STATUS_WORD = "meets" if DEPLOYMENT_READY else "does not yet meet"
INSIGHTS = [
    {
        "headline": f"Problem 1's {UPSTREAM_CHAMPION} model {DEPLOY_STATUS_WORD} the scorecard deployment bar",
        "specific": f"Score AUC {REAL_SCORE_AUC:.4f} (95% bootstrap CI [{AUC_CI_LOW:.4f}, {AUC_CI_HIGH:.4f}]); "
                    f"deployment checks {sum(1 for _, ok in deployment_checks if ok)}/{len(deployment_checks)} PASS.",
        "measurable": f"Calibration gap {MEAN_CALIBRATION_GAP:.4f} vs. <{CALIBRATION_GAP_THRESHOLD} threshold; "
                      f"split-half PSI {SPLIT_HALF_PSI:.4f} vs. <{PSI_STABILITY_THRESHOLD} threshold.",
        "achievable": "No further tuning required this cycle." if DEPLOYMENT_READY else
                      "Investigate the failing check(s) above before promoting to production; "
                      "re-run this notebook after any fix to confirm.",
        "relevant": "Directly supports real-time credit-score display in the underwriting-automation workflow.",
        "timebound": "Verdict computed fresh on every run — re-check before each deployment cycle.",
    },
    {
        "headline": f"Reduce the {thin_pct:.1%} thin-file population's reliance on application-only signal",
        "specific": f"{n_thin:,} of {N_SCORE_POP:,} real customers have zero bureau credit records and are "
                    f"scored on application fields alone.",
        "measurable": "Track the thin-file share and its mean score gap vs. bureau-scored customers on every run.",
        "achievable": "Consider Notebook 02's broader servicing-history feature set (POS_CASH/installments/"
                      "credit_card) as an alternative-data signal for this specific population in a future cycle.",
        "relevant": "Directly extends the standing instruction to integrate relevant supplementary datasets "
                    "for accuracy, applied here to the thin-file sub-population specifically.",
        "timebound": "Target: evaluate alongside the next Mega Project 1 feature-harmonization pass.",
    },
    {
        "headline": f"'{_worst_band['SCORE_BAND']}' band concentrates the real default risk",
        "specific": f"Real default rate {_worst_band['real_default_rate']:.2%} in the '{_worst_band['SCORE_BAND']}' "
                    f"band vs. {_best_band['real_default_rate']:.2%} in '{_best_band['SCORE_BAND']}'.",
        "measurable": f"${float(portfolio_by_band.get(_worst_band['SCORE_BAND'], 0.0)):,.0f} of real portfolio "
                      f"volume sits in this highest-risk band.",
        "achievable": "Route this band to enhanced manual review or tighter approval terms in the underwriting policy.",
        "relevant": "Directly supports risk-tiered decisioning across the underwriting-automation workflow.",
        "timebound": "Target: incorporate into the next underwriting policy review cycle.",
    },
]

insights_summary_df = pd.DataFrame(INSIGHTS)

csv_paths = write_csv_outputs(
    {
        "notebook_03_customer_scores": score_export_df,
        "notebook_03_band_validation": band_validation_export,
        "notebook_03_shap_feature_importance": shap_importance_df,
        "notebook_03_lime_explanations": lime_explanations_df,
        "notebook_03_calibration_by_decile": calibration_df,
        "notebook_03_insights_summary": insights_summary_df,
    },
    REPORTS_DIR,
)

word_path = build_word_report(
    REPORTS_DIR / "notebook_03_report.docx",
    title="Problem 4 — Credit Score Estimation",
    subtitle="Mega Project 1: Intelligent Underwriting & Automated Credit Decisioning",
    exec_summary=[
        f"Real PDO scorecard built on Problem 1's real {UPSTREAM_CHAMPION} PD model — {N_SCORE_POP:,} real "
        f"customers scored, score range {SCORE.min():.0f}-{SCORE.max():.0f}.",
        f"Real score discrimination: AUC {REAL_SCORE_AUC:.4f} (95% bootstrap CI [{AUC_CI_LOW:.4f}, {AUC_CI_HIGH:.4f}]).",
        f"Deployment readiness verdict: {DEPLOYMENT_VERDICT}",
        f"Thin-file population (no bureau history): {n_thin:,} ({thin_pct:.2%}) of scored customers.",
        f"SHAP + LIME explainability computed for the upstream champion model; top real driver: "
        f"{TOP_SHAP_FEATURES.iloc[0]['feature']}.",
        f"All {len(checks)} pipeline integrity checks: {sum(1 for _, ok in checks if ok)}/{len(checks)} PASS.",
    ],
    insights=INSIGHTS,
    sections=[
        {"heading": "Exploratory Data Analysis & Data Quality of the Scoring Population (SOP Stage 1B/2)",
         "paragraphs": [
             f"{len(null_counts)} of {len(FEATURE_COLS)} real scoring features have at least one missing value "
             f"(imputed via Notebook 01's fitted imputer, never dropped).",
             "IQR-based outlier counts (real): " + "; ".join(
                 f"{o['column']}={o['n_outliers_iqr']:,} ({o['pct_outliers_iqr']:.1%})" for o in OUTLIER_SUMMARY
             ) + ".",
         ],
         "image_path": eda_overview_path,
         "story": STORY_MISSING_CHART},
        {"heading": "Numeric Distributions & Outliers", "image_path": eda_distributions_path},
        {"heading": "Correlation of Scoring Features with Target", "image_path": eda_correlation_path},
        {"heading": "Real Score Distribution & Band Validation",
         "table": {"headers": ["Score Band", "Customers", "Real Default Rate", "Mean Score"],
                   "rows": [[r["SCORE_BAND"], int(r["n_customers"]), f"{r['real_default_rate']:.4f}", f"{r['mean_score']:.0f}"]
                            for _, r in band_validation.iterrows()]},
         "image_path": ARTIFACTS_DIR / "notebook_03_score_distribution.png",
         "story": STORY_BAND_CHART},
        {"heading": f"SHAP Explainability (Upstream Champion Model: {UPSTREAM_CHAMPION})",
         "paragraphs": [
             f"Computed on a real {SHAP_SAMPLE_N}-row sample of the scoring population. Top 5 real drivers by "
             f"mean |SHAP value|: {TOP_SHAP_FEATURES.head(5)[['feature', 'mean_abs_shap']].round(4).to_dict('records')}.",
         ],
         "image_path": shap_summary_path,
         "story": STORY_SHAP_CHART},
        {"heading": "LIME Explainability (Upstream Champion Model, representative real scored customers)",
         "paragraphs": [f"Real cases explained: {', '.join(_lime_cases.keys())}."],
         "table": {"headers": ["Case", "Feature Condition", "Weight"],
                   "rows": [[r["case"], r["feature_condition"], f"{r['weight']:.4f}"] for r in LIME_EXPLANATIONS]},
         "image_path": lime_panel_path,
         "story": STORY_LIME_CHART},
        {"heading": "Statistical Validation (SOP Stage 4)",
         "paragraphs": [
             f"Bootstrap 95% CI on score AUC ({N_BOOTSTRAP} resamples): [{AUC_CI_LOW:.4f}, {AUC_CI_HIGH:.4f}].",
             f"Mean calibration gap across {len(calib_summary)} PD deciles: {MEAN_CALIBRATION_GAP:.4f} "
             f"(threshold: <{CALIBRATION_GAP_THRESHOLD}).",
             f"Split-half PSI on the SCORE distribution: {SPLIT_HALF_PSI:.4f} (threshold: <{PSI_STABILITY_THRESHOLD}).",
         ],
         "story": STORY_CALIB_CHART},
        {"heading": "Portfolio & Financial Impact (real volume figures + one labeled assumption)",
         "paragraphs": [
             f"Total real scored portfolio volume (AMT_CREDIT): ${TOTAL_PORTFOLIO_VOLUME:,.0f} across "
             f"{N_SCORE_POP:,} real customers.",
             f"ASSUMPTION (illustrative manual-scoring cost, ${ASSUMPTIONS['AVG_MANUAL_SCORING_COST_PER_CUSTOMER']:.2f}/"
             f"customer): applied only to the automated-scoring count, illustrative cost avoided = "
             f"${ESTIMATED_MANUAL_SCORING_COST_AVOIDED:,.0f}. This is a labeled estimate, not a real observed figure.",
         ]},
        {"heading": "Integrity Checks",
         "table": {"headers": ["Check", "Result"],
                   "rows": [[name, "PASS" if ok else "FAIL"] for name, ok in checks]}},
    ],
)

cost_ref = assumption_ref(ASSUMPTIONS, "AVG_MANUAL_SCORING_COST_PER_CUSTOMER")
excel_path = build_excel_workbook(
    REPORTS_DIR / "notebook_03_workbook.xlsx",
    assumptions=ASSUMPTIONS,
    assumption_notes=ASSUMPTION_NOTES,
    data_sheets=[
        {"name": "Band Validation", "headers": ["Score Band", "Customers", "Real Default Rate", "Mean Score", "Portfolio AMT_CREDIT"],
         "rows": band_validation_export.values.tolist(), "highlight_col": "Real Default Rate"},
        {"name": "SHAP Feature Importance", "headers": ["Feature", "Mean |SHAP|"],
         "rows": shap_importance_df.values.tolist(), "highlight_col": "Mean |SHAP|"},
        {"name": "LIME Instance Explanations", "headers": ["Case", "Feature Condition", "Weight"],
         "rows": [[r["case"], r["feature_condition"], r["weight"]] for r in LIME_EXPLANATIONS]},
        {"name": "Calibration by Decile", "headers": ["Score Decile", "Mean Predicted PD", "Actual Rate", "N"],
         "rows": calibration_df.values.tolist(), "highlight_col": "Actual Rate"},
        {"name": "Integrity Checks", "headers": ["Check", "Result"],
         "rows": [[name, "PASS" if ok else "FAIL"] for name, ok in checks]},
    ],
    formula_sheet={
        "name": "Financial Impact",
        "rows": [
            ("Total Scored Portfolio Volume ($)", TOTAL_PORTFOLIO_VOLUME),
            ("Customers Scored", N_SCORE_POP),
            ("Thin-File Customers", n_thin),
            ("Est. Manual-Scoring Cost Avoided ($, illustrative)", f"={N_SCORE_POP}*{cost_ref}"),
        ],
    },
    insights_sheet={"name": "Insights & SMART Actions", "items": INSIGHTS},
)

# --- Real alternate "slicer" views ------------------------------------------
band_view_default_rate = {"key": "default_rate", "label": "By Real Default Rate",
                           "labels": band_validation["SCORE_BAND"].astype(str).tolist(),
                           "datasets": [{"label": "Real Default Rate", "data": band_validation["real_default_rate"].round(4).tolist(),
                                         "backgroundColor": _palette(len(band_validation))}]}
band_view_count = {"key": "count", "label": "By Customer Count",
                    "labels": band_validation["SCORE_BAND"].astype(str).tolist(),
                    "datasets": [{"label": "Customers", "data": band_validation["n_customers"].tolist(),
                                  "backgroundColor": _palette(len(band_validation))}]}

shap_view = {"key": "shap", "label": "SHAP Feature Importance",
             "labels": TOP_SHAP_FEATURES["feature"].tolist(),
             "datasets": [{"label": "Mean |SHAP|", "data": TOP_SHAP_FEATURES["mean_abs_shap"].round(4).tolist(),
                           "backgroundColor": _palette(len(TOP_SHAP_FEATURES))}]}

_lime_first_case = next(iter(_lime_cases.keys())) if _lime_cases else None
_lime_first_rows = [r for r in LIME_EXPLANATIONS if r["case"] == _lime_first_case] if _lime_first_case else []
lime_view = {"key": "lime", "label": f"LIME — {_lime_first_case}" if _lime_first_case else "LIME",
             "labels": [r["feature_condition"] for r in _lime_first_rows],
             "datasets": [{"label": "Local Weight", "data": [round(r["weight"], 4) for r in _lime_first_rows],
                           "backgroundColor": [VIVID_PALETTE[2] if r["weight"] >= 0 else VIVID_PALETTE[7]
                                                for r in _lime_first_rows]}]}

top_missing_5 = top_missing.head(5)
missing_view_15 = {"key": "top15", "label": f"Top {len(top_missing)} Columns",
                    "labels": top_missing["column"].tolist(),
                    "datasets": [{"label": "% Missing", "data": (top_missing["pct_null"] * 100).round(2).tolist(),
                                  "backgroundColor": _palette(len(top_missing))}]} if len(top_missing) else None
missing_view_5 = {"key": "top5", "label": "Top 5 Columns",
                   "labels": top_missing_5["column"].tolist(),
                   "datasets": [{"label": "% Missing", "data": (top_missing_5["pct_null"] * 100).round(2).tolist(),
                                 "backgroundColor": _palette(len(top_missing_5))}]} if len(top_missing_5) else None

SAMPLE_N = min(200, len(score_export_df))
sample_df = (
    score_export_df.sample(n=SAMPLE_N, random_state=SEED)
    .sort_values("CREDIT_SCORE", ascending=False)
    .round({"PD": 4, "CREDIT_SCORE": 0})
)

html_charts = [
    {"id": "scoreChart", "title": "Real Score Distribution", "type": "bar",
     "labels": band_view_count["labels"], "datasets": band_view_count["datasets"], "showLegend": False,
     "views": [band_view_count, band_view_default_rate], "story": STORY_SCORE_CHART},
    {"id": "bandChart", "title": "Real Default Rate by Score Band", "type": "bar",
     "labels": band_view_default_rate["labels"], "datasets": band_view_default_rate["datasets"], "showLegend": False,
     "views": [band_view_default_rate, band_view_count], "story": STORY_BAND_CHART},
    {"id": "shapChart", "title": f"SHAP Feature Importance — Upstream Champion ({UPSTREAM_CHAMPION})", "type": "bar",
     "labels": shap_view["labels"], "datasets": shap_view["datasets"], "showLegend": False,
     "story": STORY_SHAP_CHART},
    {"id": "limeChart", "title": f"LIME — {_lime_first_case or 'Champion'} (real scored customer)", "type": "bar",
     "labels": lime_view["labels"], "datasets": lime_view["datasets"], "showLegend": False,
     "story": STORY_LIME_CHART},
    {"id": "calibChart", "title": "Calibration by PD Decile (real)", "type": "line",
     "labels": [str(int(d)) for d in calib_summary["decile"]],
     "datasets": [
         {"label": "Mean Predicted PD", "data": calib_summary["mean_predicted"].round(4).tolist(), "borderColor": VIVID_PALETTE[0], "backgroundColor": VIVID_PALETTE[0], "fill": False},
         {"label": "Actual Default Rate", "data": calib_summary["actual_rate"].round(4).tolist(), "borderColor": VIVID_PALETTE[7], "backgroundColor": VIVID_PALETTE[7], "fill": False},
     ],
     "note": f"Mean calibration gap: {MEAN_CALIBRATION_GAP:.4f}", "story": STORY_CALIB_CHART},
    {"id": "thinFileChart", "title": "Real Bureau-Coverage Split", "type": "doughnut",
     "labels": ["Bureau-Scored", "Thin-File (App-Only)"],
     "datasets": [{"data": [N_SCORE_POP - n_thin, n_thin], "backgroundColor": [VIVID_PALETTE[0], VIVID_PALETTE[3]]}],
     "story": STORY_MISSING_CHART},
]
if missing_view_15 is not None:
    html_charts.append(
        {"id": "missingChart", "title": "Top Scoring Features by Real Missing-Value %", "type": "bar",
         "labels": missing_view_15["labels"], "datasets": missing_view_15["datasets"], "showLegend": False,
         "views": [missing_view_15, missing_view_5], "story": STORY_MISSING_CHART}
    )

html_path = build_html_dashboard(
    REPORTS_DIR / "notebook_03_dashboard.html",
    title="Problem 4 — Credit Score Estimation",
    subtitle=f"Upstream model: {UPSTREAM_CHAMPION} | Score AUC: {REAL_SCORE_AUC:.4f} | {DEPLOYMENT_VERDICT}",
    kpi_cards=[
        {"label": "Customers Scored", "value": f"{N_SCORE_POP:,}"},
        {"label": "Score AUC", "value": f"{REAL_SCORE_AUC:.4f}"},
        {"label": "Median Score", "value": f"{np.median(SCORE):.0f}"},
        {"label": "Thin-File Rate", "value": f"{thin_pct:.1%}"},
        {"label": "Portfolio Volume", "value": f"${TOTAL_PORTFOLIO_VOLUME:,.0f}"},
        {"label": "Integrity Checks", "value": f"{sum(1 for _, ok in checks if ok)}/{len(checks)} PASS"},
        {"label": "Top SHAP Driver", "value": TOP_SHAP_FEATURES.iloc[0]["feature"]},
    ],
    insights=INSIGHTS,
    charts=html_charts,
    data_table={
        "title": f"Sampled Real Scored Customers ({SAMPLE_N} of {len(score_export_df):,} rows)",
        "columns": ["SK_ID_CURR", "PD", "CREDIT_SCORE", "SCORE_BAND", "TARGET"],
        "rows": sample_df[["SK_ID_CURR", "PD", "CREDIT_SCORE", "SCORE_BAND", "TARGET"]].values.tolist(),
        "filter_column": "SCORE_BAND",
    },
)
print(f"[REPORTING] Real reporting package written: reports/{word_path.name}, reports/{excel_path.name}, "
      f"reports/{html_path.name}, plus {len(csv_paths)} CSV file(s) (all under decision_engine/reports/).")

# ---------------------------------------------------------------------------
# SECTION 12 — Save artifacts + governance stamp (SOP Stage 6: Production
# Packaging & Governance) — idempotent: overwrite in place, fixed paths
# ---------------------------------------------------------------------------
summary = {
    "notebook": "03_credit_score_estimation",
    "mega_project": "Mega Project 1 - Intelligent Underwriting & Automated Credit Decisioning",
    "problem": "Problem 4 - Credit Score Estimation",
    "upstream_model_notebook": "01_credit_default_prediction",
    "upstream_champion": UPSTREAM_CHAMPION,
    "random_seed": SEED,
    "n_customers_scored": int(len(pdf)),
    "eda_data_quality": {
        "n_columns_with_missing": int(len(null_counts)),
        "top_5_missing_pct": {row["column"]: round(float(row["pct_null"]), 4) for _, row in null_counts.head(5).iterrows()},
        "iqr_outliers": OUTLIER_SUMMARY,
        "top_target_correlations": target_corr.abs().sort_values(ascending=False).head(3).round(4).to_dict(),
        "eda_chart_files": [p.name for p in EDA_CHART_PATHS],
    },
    "performance_config": {
        "logical_cores_detected": TOTAL_THREADS,
        "total_ram_gb_detected": TOTAL_RAM_GB,
        "cpu_thread_ceiling_applied": CPU_CEILING_THREADS,
        "ram_ceiling_gb": RAM_CEILING_GB,
        "cpu_affinity_pinned_cores": PERF.get("logical_cores"),
        "parquet_cache_dir": str(PARQUET_CACHE_DIR.name),
    },
    "explainability": {
        "shap_sample_size": SHAP_SAMPLE_N,
        "shap_top_10_features": shap_importance_df.head(10).round(4).to_dict("records"),
        "lime_cases_explained": list(_lime_cases.keys()),
        "lime_explanation_count": len(LIME_EXPLANATIONS),
    },
    "scaling_assumptions": {"base_score": BASE_SCORE, "base_odds": BASE_ODDS, "pdo": PDO,
                             "source": "standard credit-scorecard convention (FICO-style), not derived from this data"},
    "score_distribution": {
        "min": float(SCORE.min()), "p25": float(np.percentile(SCORE, 25)),
        "median": float(np.median(SCORE)), "p75": float(np.percentile(SCORE, 75)),
        "max": float(SCORE.max()),
    },
    "band_validation": band_validation.to_dict(orient="records"),
    "score_monotonicity_holds": bool(is_monotonic_decreasing),
    "thin_file_customers": n_thin,
    "thin_file_pct": thin_pct,
    "statistical_validation": {
        "bootstrap_resamples": N_BOOTSTRAP,
        "score_auc": REAL_SCORE_AUC,
        "score_auc_ci_95": [AUC_CI_LOW, AUC_CI_HIGH],
        "mean_calibration_gap": MEAN_CALIBRATION_GAP,
        "split_half_psi": SPLIT_HALF_PSI,
        "deployment_checks": {name: ok for name, ok in deployment_checks},
        "failed_deployment_checks": _failed_deployment_checks,
        "monotonicity_detail": _monotonicity_detail,
        "deployment_verdict": DEPLOYMENT_VERDICT,
        "note": "deployment_checks (statistical robustness) is a separate check family from "
                "integrity_checks (structural pipeline sanity) below -- see deployment_verdict "
                "for which specific statistical check(s), if any, failed on this run. "
                "score_monotonicity_holds uses a real, Bonferroni-corrected two-proportion "
                "z-test per adjacent band pair (see monotonicity_detail above and "
                "src/utils/stats_checks.py) -- as of CHANGELOG [1.0.2], not a strict "
                "zero-tolerance ordering check.",
    },
    "financial_impact": {
        "total_scored_portfolio_volume_usd": TOTAL_PORTFOLIO_VOLUME,
        "assumptions": ASSUMPTIONS,
        "estimated_manual_scoring_cost_avoided_usd_illustrative": ESTIMATED_MANUAL_SCORING_COST_AVOIDED,
    },
    "integrity_checks": {n: bool(ok) for n, ok in checks},
    "reporting_artifacts": ["notebook_03_report.docx", "notebook_03_workbook.xlsx",
                             "notebook_03_dashboard.html"] + [f"{stem}.csv" for stem in csv_paths],
    "sop_stage_reached": "6 - Production Packaging & Governance",
    "runtime_seconds": round(time.time() - T0, 1),
}
with open(ARTIFACTS_DIR / "notebook_03_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"[DONE] Notebook 03 complete in {summary['runtime_seconds']}s using a {CPU_CEILING_THREADS}-thread "
      f"WARP ceiling. {len(pdf):,} customers scored using Notebook 01's real {UPSTREAM_CHAMPION} model. "
      f"Deployment verdict: {DEPLOYMENT_VERDICT}.")
